In [2]:
"""
Phase 1 profiling script: loads TinyLlama in 4-bit, runs generation under
the PyTorch profiler, and prints per-shape timing for the GEMM kernels.

Written defensively against transformers/torch API drift (attribute names
like cuda_time_total -> device_time_total have changed across versions).
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_model(model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    """Load a 4-bit quantized causal LM + tokenizer.

    Returns:
        (model, tokenizer)
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
    )
    return model, tokenizer


def run_profiled_generation(model, tokenizer, prompt: str = "Explain how a GPU works.",
                             max_new_tokens: int = 50):
    """Run model.generate() under the PyTorch profiler.

    Returns:
        prof: the completed profiler context (already exited)
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
    ) as prof:
        model.generate(**inputs, max_new_tokens=max_new_tokens)

    return prof


def _get_time_total_us(evt) -> float:
    """Return this event's total device time in microseconds.

    Handles the cuda_time_total -> device_time_total rename across
    torch versions without crashing.
    """
    for attr in ("device_time_total", "cuda_time_total", "self_device_time_total"):
        val = getattr(evt, attr, None)
        if val is not None:
            return val
    return 0.0


def summarize_gemm_events(prof, name_filter: str = "gemm_4bit"):
    """Print per-shape call count and total time for events matching name_filter.

    If shapes come back empty (bitsandbytes kernels sometimes don't report
    them), prints a warning so you know to fall back to reading layer dims
    directly from the model instead.
    """
    events = prof.key_averages(group_by_input_shape=True)
    matched = [e for e in events if name_filter in e.key]

    if not matched:
        print(f"No events matched '{name_filter}'. Available event names:")
        for e in events[:15]:
            print(f"  {e.key}")
        return

    saw_empty_shape = False
    for evt in matched:
        shape = getattr(evt, "input_shapes", None)
        if not shape:
            saw_empty_shape = True
        time_ms = _get_time_total_us(evt) / 1000
        print(f"shape={shape}  calls={evt.count}  time_total={time_ms:.3f}ms")

    if saw_empty_shape:
        print(
            "\n[warning] Some events had no shape info. "
            "Run print_linear4bit_shapes(model) to get shapes directly "
            "from the model's layer definitions instead."
        )


def print_linear4bit_shapes(model):
    """Fallback: print in/out features for every 4-bit Linear layer in the model.

    Use this when the profiler doesn't report input_shapes for quantized
    kernels. Attention projections (q/k/v/o_proj) and FFN projections
    (gate/up/down_proj) will show distinctly different shapes.
    """
    for name, module in model.named_modules():
        if "Linear4bit" in type(module).__name__:
            print(f"{name}: in={module.in_features}, out={module.out_features}")


if __name__ == "__main__":
    model, tokenizer = load_model()
    prof = run_profiled_generation(model, tokenizer)

    print("\n--- GEMM events by shape ---")
    summarize_gemm_events(prof)

    print("\n--- Linear4bit layer shapes (fallback reference) ---
    print_linear4bit_shapes(model)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- GEMM events by shape ---
shape=[]  calls=110  time_total=24.742ms
shape=[]  calls=44  time_total=1.819ms

[warning] Some events had no shape info. Run print_linear4bit_shapes(model) to get shapes directly from the model's layer definitions instead.

--- Linear4bit layer shapes (fallback reference) ---
model.layers.0.self_attn.q_proj: in=2048, out=2048
model.layers.0.self_attn.k_proj: in=2048, out=256
model.layers.0.self_attn.v_proj: in=2048, out=256
model.layers.0.self_attn.o_proj: in=2048, out=2048
model.layers.0.mlp.gate_proj: in=2048, out=5632
model.layers.0.mlp.up_proj: in=2048, out=5632
model.layers.0.mlp.down_proj: in=5632, out=2048
model.layers.1.self_attn.q_proj: in=2048, out=2048
model.layers.1.self_attn.k_proj: in=2048, out=256
model.layers.1.self_attn.v_proj: in=2048, out=256
model.layers.1.self_attn.o_proj: in=2048, out=2048
model.layers.1.mlp.gate_proj: in=2048, out=5632
model.layers.1.mlp.up_proj: in=2048, out=5632
model.layers.1.mlp.down_proj: in=5632, out=2048
mode